# Machine Learning for Cryo-Electron Tomography (Cryo-ET)

**Module:** `ml` — Week 13, ML for Structural Biology
**Prerequisites:** Weeks 1–10 (regression/classification fundamentals, CNNs)

## Learning objectives

By the end of this notebook you will be able to:

1. Explain what a cryo-ET subtomogram is and why it is a hard ML data type (low SNR, missing wedge, 3D volume).
2. Build a synthetic subtomogram dataset to prototype models without needing lab access to real cryo-ET data.
3. Train a 3D CNN classifier to distinguish macromolecular complexes ("particle classification").
4. Evaluate the classifier with metrics appropriate for imbalanced, noisy scientific data.
5. Connect this toy pipeline to real tools used in the lab (`aitom`, `xulabs/edu/cryoem`).

## Background

Cryo-electron tomography reconstructs 3D volumes ("tomograms") of frozen-hydrated cells or purified
complexes. A key downstream task is **subtomogram classification**: given a small 3D crop ("subtomogram")
around a detected particle, decide which macromolecular complex (ribosome, proteasome, membrane protein,
etc.) it contains — or whether it is background/noise.

This is a genuinely hard vision problem because:

- **Very low signal-to-noise ratio (SNR)** — often below 0.1, due to radiation-damage limits on electron dose.
- **Missing wedge artifacts** — tomograms are reconstructed from a limited tilt range, so information is
  missing along one direction in Fourier space, causing anisotropic blurring.
- **Class imbalance** — most subtomograms are background; true particles of a given type may be rare.
- **3D, not 2D** — standard image-classification intuition from 2D CNNs must be extended to 3D convolutions,
  which are more expensive and more data-hungry.

We will not use real experimental data here (that lives in lab-internal storage and is covered in the
`cryoem/` module and `aitom` codebase). Instead we build a **synthetic subtomogram dataset** that mimics
the key difficulties above, so the notebook runs anywhere on CPU in a few minutes and the concepts transfer
directly when you move to real data.


## 1. Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 2. Building a synthetic subtomogram dataset

We simulate three classes of 3D subtomograms, each `32 x 32 x 32` voxels:

- **Class 0 — background**: pure correlated noise, no structure.
- **Class 1 — "sphere" complex**: a roughly spherical density blob (stand-in for a compact complex like a
  proteasome).
- **Class 2 — "rod" complex**: an elongated density (stand-in for a filament or elongated complex).

We then corrupt every sample with:

- Gaussian noise at a low SNR (mimicking cryo-ET's dose limits).
- A **missing-wedge** mask applied in Fourier space (zeroing out a wedge of frequencies along one axis),
  which is the single most distinctive cryo-ET artifact.


In [ ]:
def make_sphere(size=32, radius=8, center_jitter=3):
    c = size // 2 + np.random.randint(-center_jitter, center_jitter + 1, size=3)
    zz, yy, xx = np.meshgrid(np.arange(size), np.arange(size), np.arange(size), indexing="ij")
    r = radius + np.random.uniform(-1.5, 1.5)
    dist = np.sqrt((zz - c[0])**2 + (yy - c[1])**2 + (xx - c[2])**2)
    vol = np.clip(1.0 - dist / r, 0, 1)
    return vol.astype(np.float32)

def make_rod(size=32, length=20, radius=4, center_jitter=3):
    c = size // 2 + np.random.randint(-center_jitter, center_jitter + 1, size=3)
    zz, yy, xx = np.meshgrid(np.arange(size), np.arange(size), np.arange(size), indexing="ij")
    # elongated along a random axis by stretching one coordinate
    axis = np.random.choice(["z", "y", "x"])
    if axis == "z":
        dist = np.sqrt(((zz - c[0]) / (length / 2))**2 + (yy - c[1])**2 + (xx - c[2])**2) * radius
    elif axis == "y":
        dist = np.sqrt((zz - c[0])**2 + ((yy - c[1]) / (length / 2))**2 + (xx - c[2])**2) * radius
    else:
        dist = np.sqrt((zz - c[0])**2 + (yy - c[1])**2 + ((xx - c[2]) / (length / 2))**2) * radius
    vol = np.clip(1.0 - dist / radius, 0, 1)
    return vol.astype(np.float32)

def make_background(size=32):
    return np.zeros((size, size, size), dtype=np.float32)

def apply_missing_wedge(vol, wedge_deg=30):
    """Zero out a wedge of frequencies along the z-tilt axis, mimicking a limited-tilt-range reconstruction."""
    F3 = np.fft.fftshift(np.fft.fftn(vol))
    size = vol.shape[0]
    zz, yy, xx = np.meshgrid(
        np.linspace(-1, 1, size), np.linspace(-1, 1, size), np.linspace(-1, 1, size), indexing="ij"
    )
    angle = np.degrees(np.arctan2(xx, zz + 1e-8))
    mask = np.abs(angle) < (90 - wedge_deg)
    F3_masked = F3 * (~mask)  # zero out the "missing wedge" region
    vol_mw = np.real(np.fft.ifftn(np.fft.ifftshift(F3_masked)))
    return vol_mw.astype(np.float32)

def add_noise(vol, snr=0.1):
    signal_power = np.mean(vol**2) + 1e-8
    noise_power = signal_power / snr
    noise = np.random.normal(0, np.sqrt(noise_power), size=vol.shape).astype(np.float32)
    return vol + noise

def make_sample(label, size=32):
    if label == 0:
        vol = make_background(size)
        vol = add_noise(vol, snr=1.0)  # pure noise, no missing-wedge needed
    else:
        vol = make_sphere(size) if label == 1 else make_rod(size)
        vol = apply_missing_wedge(vol, wedge_deg=30)
        vol = add_noise(vol, snr=0.15)
    # normalize per-sample, as is standard for cryo-ET preprocessing
    vol = (vol - vol.mean()) / (vol.std() + 1e-8)
    return vol


### Visualize a few examples

Central-slice views of one sample per class — this is how subtomograms are typically inspected by eye.
Notice how noisy and directionally-blurred the "sphere" and "rod" classes look once the missing wedge and
noise are applied; this is intentionally close to what real cryo-ET data looks like.


In [ ]:
class_names = ["background", "sphere", "rod"]
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for label, ax in zip([0, 1, 2], axes):
    vol = make_sample(label)
    ax.imshow(vol[vol.shape[0] // 2], cmap="gray")
    ax.set_title(class_names[label])
    ax.axis("off")
plt.suptitle("Central z-slice of one synthetic subtomogram per class")
plt.tight_layout()
plt.show()


## 3. PyTorch `Dataset` and `DataLoader`

In [ ]:
class SubtomogramDataset(Dataset):
    def __init__(self, n_per_class=200, size=32):
        self.size = size
        self.samples = []
        self.labels = []
        for label in [0, 1, 2]:
            for _ in range(n_per_class):
                self.samples.append(make_sample(label, size))
                self.labels.append(label)
        self.samples = np.stack(self.samples)  # (N, D, H, W)
        self.labels = np.array(self.labels, dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        vol = self.samples[idx][None, ...]  # add channel dim -> (1, D, H, W)
        return torch.from_numpy(vol), self.labels[idx]

train_ds = SubtomogramDataset(n_per_class=200)
val_ds = SubtomogramDataset(n_per_class=50)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

print(f"Train set: {len(train_ds)} subtomograms, Val set: {len(val_ds)} subtomograms")


## 4. A small 3D CNN classifier

We use a compact 3D CNN (three `Conv3d` blocks + global average pooling). Real cryo-ET pipelines
(e.g. `aitom`'s classification modules) use similar but larger architectures, sometimes with additional
tricks like missing-wedge-aware augmentation or rotation-invariant pooling — flagged below as extensions.


In [ ]:
class Simple3DCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.conv1 = nn.Conv3d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv3d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv3d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool3d(2)
        self.gap = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 32 -> 16
        x = self.pool(F.relu(self.conv2(x)))   # 16 -> 8
        x = self.pool(F.relu(self.conv3(x)))   # 8 -> 4
        x = self.gap(x).flatten(1)             # (B, 64)
        return self.fc(x)

model = Simple3DCNN(num_classes=3).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model has {n_params:,} parameters")


## 5. Training loop

In [ ]:
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, total_correct, total_n = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for vols, labels in loader:
            vols, labels = vols.to(device), labels.to(device)
            logits = model(vols)
            loss = F.cross_entropy(logits, labels)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * vols.size(0)
            total_correct += (logits.argmax(1) == labels).sum().item()
            total_n += vols.size(0)
    return total_loss / total_n, total_correct / total_n

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

N_EPOCHS = 12
for epoch in range(N_EPOCHS):
    tr_loss, tr_acc = run_epoch(model, train_loader, optimizer)
    va_loss, va_acc = run_epoch(model, val_loader, optimizer=None)
    history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss); history["val_acc"].append(va_acc)
    print(f"Epoch {epoch+1:2d}/{N_EPOCHS} | train loss {tr_loss:.3f} acc {tr_acc:.3f} "
          f"| val loss {va_loss:.3f} acc {va_acc:.3f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.show()


## 6. Evaluation beyond accuracy

Real subtomogram datasets are **class-imbalanced** (background patches vastly outnumber true particles),
so accuracy alone is misleading. We compute a per-class confusion matrix and precision/recall, which is
the standard way particle-picking pipelines are evaluated in the literature.


In [ ]:
from collections import Counter

def confusion_matrix(model, loader, n_classes=3):
    model.eval()
    cm = np.zeros((n_classes, n_classes), dtype=int)
    with torch.no_grad():
        for vols, labels in loader:
            vols = vols.to(device)
            preds = model(vols).argmax(1).cpu().numpy()
            for t, p in zip(labels.numpy(), preds):
                cm[t, p] += 1
    return cm

cm = confusion_matrix(model, val_loader)
print("Confusion matrix (rows = true, cols = predicted):")
print(class_names)
print(cm)

for i, name in enumerate(class_names):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    print(f"{name:12s}  precision={precision:.2f}  recall={recall:.2f}")


## 7. From this toy pipeline to real cryo-ET

This notebook deliberately simplifies cryo-ET to keep it runnable anywhere. To move toward real research
use, the main gaps to close are:

1. **Real data & particle picking.** Subtomograms here are hand-generated; in practice they come from a
   picking step (template matching, difference-of-Gaussian, or a learned picker) applied to a full
   reconstructed tomogram. See the `cryoem/` module and `aitom`'s `pick/` submodule.
2. **Realistic missing-wedge modeling.** We used a simple angular Fourier mask; real acquisition geometry
   (tilt range, tilt scheme) determines the exact wedge shape, and some architectures explicitly model it
   (e.g. wedge-aware pooling or Fourier-domain layers).
3. **Class imbalance and hard negatives.** Real background patches are far more numerous and more varied
   than our synthetic noise; techniques like hard-negative mining and focal loss are common.
4. **Rotation invariance.** Particles appear at random 3D orientations. Data augmentation with random 3D
   rotations (or rotation-equivariant architectures) substantially improves real-world performance —
   try adding this as an exercise below.
5. **Larger, deeper 3D architectures** (3D ResNets, or 2D-projection ensembles) and self-supervised
   pretraining are active areas covered in the lab's ongoing subtomogram classification work.

### Suggested exercises

- Add random 3D rotation augmentation to `SubtomogramDataset.__getitem__` and re-train. Does validation
  accuracy improve?
- Sweep the noise `snr` parameter in `add_noise` and plot accuracy vs. SNR — this reproduces, in miniature,
  the core difficulty curve of real cryo-ET classification.
- Replace `Simple3DCNN` with a 3D ResNet block and compare parameter efficiency vs. accuracy.
- Read one of the papers referenced in `aitom`'s classification module and compare its architecture choices
  to the ones discussed here.

## References

- Xu, M. et al. Machine learning approaches for cryo-electron tomography. (see `aitom/doc/publications.md`
  for the lab's full publication list on this topic.)
- `xulabs/aitom` — production codebase for tomogram analysis, including real particle picking and
  classification pipelines.
